In [1]:
# lets load the pdf using UnStructured

from langchain_community.document_loaders import UnstructuredPDFLoader
  


In [2]:
loader = UnstructuredPDFLoader(
    file_path="../data/test.pdf",
    mode="paged",
    strategy="auto",
    encoding="utf-8",
)

In [3]:
docs = loader.load()

print("Documenyts loaded successfully")
print(f"Loaded {len(docs)} documents")

d:\ML(ExtraClass Project)\RAG_PROJECT\PDF-RAG-Chatbot-Ask-Anything-About-a-PDF\.pdfchatenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`mode='paged'` is deprecated in favor of the 'by_page' chunking strategy. Learn more about chunking here: https://docs.unstructured.io/open-source/core-functionality/chunking


Documenyts loaded successfully
Loaded 12 documents


In [4]:
for i,doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)  # Print the first 500 characters of each document
    print("\n" + "="*50 + "\n")  # Separator between documents

Document 1:
AN887 APPLICATION NOTE

MICROCONTROLLERS MADE EASY by Microcontroller Division Applications

WHAT IS A MICROCONTROLLER?

A few years ago, system control functions were implemented using logic components and were usually large, heavy boxes. Later on, microprocessors were used and the entire con- troller could fit onto a small circuit board. As the process of miniaturization continued, all of the components needed for a controller were built right onto one chip. By only including the fea- tures specific to the task, cost is relatively low.

A typical microcontroller has bit manipulation instructions, easy7 and direct access to I/O, and quick and efficient interrupt processing. Therefore, a microcontroller is a highly integrated device which includes, on one chip, all or most of the parts needed to perform an application control function.

Microcontrollers come in many varieties. Depending on the power and features that are needed, customers might choose a 4, 8, 16, or 32 bit 

In [5]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [6]:
child_spilitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,)

# Split the documents into smaller chunks

In [7]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,    
    chunk_overlap=200
)

In [8]:
from langchain.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\arunp\AppData\Local\Temp\ipykernel_26352\2246710904.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


In [9]:
from langchain.vectorstores import Chroma

vectorstore = Chroma(
    collection_name="pdf_chunks",
    embedding_function=embedding_model,
    persist_directory="../chroma_db"
)


C:\Users\arunp\AppData\Local\Temp\ipykernel_26352\1065877952.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [10]:
# Create ParentDocumentRetriever
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore


In [11]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=InMemoryStore(),
    child_splitter=child_spilitter,
    parent_splitter=parent_splitter,
)




In [12]:
from langchain_community.vectorstores.utils import filter_complex_metadata
from langchain_core.documents import Document

# Step 1: Ensure we extract only Document objects if tuples are present
# (e.g., [(doc1, meta1), (doc2, meta2)] → [doc1, doc2])
pure_docs = [doc if isinstance(doc, Document) else doc[0] for doc in docs]

# Step 2: Filter out complex metadata
filtered_docs = filter_complex_metadata(pure_docs)

# Step 3: Add filtered documents to ParentDocumentRetriever
retriever.add_documents(filtered_docs)


In [13]:
# let's test the retriever
query = "What is the main topic of the document?"
results = retriever.get_relevant_documents(query)
for i, result in enumerate(results):
    print(f"Result {i+1}:")
    print(result.page_content)  # Print the content of each result
    print("\n" + "="*50 + "\n")  # Separator between results

Result 1:
MICROCONTROLLERS MADE EASY

“THE PRESENT NOTE WHICH IS FOR GUIDANCE ONLY AIMS AT PROVIDING CUSTOMERS WITH INFORMATION REGARDING THEIR PRODUCTS IN ORDER FOR THEM TO SAVE TIME. AS A RESULT, STMICROELECTRONICS SHALL NOT BE HELD LIABLE FOR ANY DIRECT, INDIRECT OR CONSEQUENTIAL DAMAGES WITH RESPECT TO ANY CLAIMS ARISING FROM THE CONTENT OF SUCH A NOTE AND/OR THE USE MADE BY CUSTOMERS OF THE INFORMATION CONTAINED HEREIN IN CONNEXION WITH THEIR PRODUCTS.”




C:\Users\arunp\AppData\Local\Temp\ipykernel_26352\76304010.py:3: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  results = retriever.get_relevant_documents(query)


In [14]:
from langchain_core.prompts import PromptTemplate

In [15]:
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a helpful assistant trained to answer questions from a PDF document.
Use only the information provided in the context below to answer the user's question.
If the answer is not present in the context, politely say you don't know.

Context:
{context}

User Question:
{question}

Answer in a clear and professional tone:
"""
)

In [ ]:
from dotenv import load_dotenv
load_dotenv()

True

In [25]:
from langchain_groq import ChatGroq

llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.2)

In [26]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()

In [27]:
from langchain.schema.runnable import  RunnableSequence, RunnableLambda, RunnablePassthrough

In [28]:
# Step 3a: Context retriever
retriever_chain = RunnableLambda(lambda x: retriever.get_relevant_documents(x["question"]))

In [29]:
# Step 3b: Prompt input assembler
format_inputs = RunnableLambda(lambda x: {
    "context": "\n\n".join([doc.page_content for doc in x["context"]]),
    "question": x["question"]
})

In [30]:
# Step 3c: Build the sequence
rag_chain = RunnableSequence(
    {
        "question": RunnablePassthrough(),     # Keep original question
        "context": retriever_chain             # Retrieve context
    },
    format_inputs,                             # Join into prompt inputs
    prompt_template,                           # Format prompt
    llm,
    parser# Run LLM
)

In [31]:
question = "What is CAN Principle?"

response = rag_chain.invoke({"question": question})

print("🤖 Chatbot says:\n", response)


🤖 Chatbot says:
 The CAN Principle is explained in Figure 13 of the document. According to the figure, the CAN Principle is divided into different systems, including INTER SYSTEM, FAST SPEED, GATEWAY, SLOW SPEED, and COMFORT, among others.


In [32]:
question = "what is the main topic of the document?"

response = rag_chain.invoke({"question": question})

print("🤖 Chatbot says:\n", response)

🤖 Chatbot says:
 The main topic of the document appears to be "Microcontrollers Made Easy" and it serves as a guidance note for customers regarding their products.


In [33]:
%pip install grandalf``

graph = rag_chain.get_graph()
graph.print_ascii()  # Print ASCII representation

Note: you may need to restart the kernel to use updated packages.
      +---------------------------------+            
      | Parallel<question,context>Input |            
      +---------------------------------+            
              ***             ***                    
            **                   **                  
          **                       **                
+-------------+          +-------------------------+ 
| Passthrough |          | ParentDocumentRetriever | 
+-------------+          +-------------------------+ 
              ***             ***                    
                 **         **                       
                   **     **                         
     +----------------------------------+            
     | Parallel<question,context>Output |            
     +----------------------------------+            
                       *                             
                       *                             
                


[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: 'grandalf``': Expected end or semicolon (after name and no valid version specifier)
    grandalf``
            ^
